In [1]:
import pandas as pd 
import numpy as np 
import os
import cv2
import matplotlib.pyplot as plt
import warnings

from tensorflow.keras.layers import Input, Lambda, Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.preprocessing import image, image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow import keras
import tensorflow 

from tensorflow.keras.applications.vgg16 import VGG16 # VGG16
from tensorflow.keras.applications.vgg19 import VGG19 # VGG19
from tensorflow.keras.applications.resnet50 import ResNet50 # ResNet50
from tensorflow.keras.applications import ResNet101 # ResNet 101
from tensorflow.keras.applications.xception import Xception # Xception
from tensorflow.keras.applications.mobilenet import MobileNet # MobileNet
from tensorflow.keras.applications.densenet import DenseNet169 # DenseNet169
from tensorflow.keras.applications.densenet import DenseNet121 # DenseNet121
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2 # MobileNetV2
from tensorflow.keras.applications.inception_v3 import InceptionV3 # InceptionV3


import scipy
print("Num GPUs Available: ", len(tensorflow.config.list_physical_devices('GPU')))


# Set the seed value for experiment reproduci.bility.
seed = 32
tensorflow.random.set_seed(seed)
np.random.seed(seed)
# Turn off warnings for cleaner looking notebook
warnings.simplefilter('ignore')

Num GPUs Available:  0


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20,
    validation_split=0.2
)

train_dataset = train_datagen.flow_from_directory(
    directory='/kaggle/input/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/Alzheimer_s Dataset/train/',
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='training'
)

validation_dataset = train_datagen.flow_from_directory(
    directory='/kaggle/input/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/Alzheimer_s Dataset/train/',
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='validation'
)

Found 5121 images belonging to 4 classes.
Found 1279 images belonging to 4 classes.


In [3]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)

callbacks = [early_stopping, model_checkpoint]


In [4]:
def train_model(base_model, train_data, val_data, epochs=50):
    base_model.trainable = False  # Freezing the layers
    inputs = Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = Flatten()(x)
    outputs = Dense(4, activation='softmax')(x)
    model = Model(inputs, outputs)

    model.compile(optimizer='adam',
                  loss=tensorflow.losses.CategoricalCrossentropy(),
                  metrics=[keras.metrics.AUC(name='auc'), 'acc'])

    model.fit(train_data, epochs=epochs, validation_data=val_data, callbacks=callbacks)
    return model


In [5]:
vgg19_model = VGG19(input_shape=(224, 224, 3), weights='imagenet', include_top=False)
model_vgg19 = train_model(vgg19_model, train_dataset, validation_dataset)

vgg16_model = VGG16(input_shape=(224, 224, 3), weights='imagenet', include_top=False)
model_vgg16 = train_model(vgg16_model, train_dataset, validation_dataset)

resnet50_model = ResNet50(input_shape=(224, 224, 3), weights='imagenet', include_top=False)
model_resnet50 = train_model(resnet50_model, train_dataset, validation_dataset)



80150528/80134624 [==============================] - 2s 0us/step
Epoch 1/50
321/321 [==============================] - 101s 280ms/step - loss: 1.1748 - auc: 0.8133 - acc: 0.5427 - val_loss: 2.1028 - val_auc: 0.7102 - val_acc: 0.4793
Epoch 2/50
321/321 [==============================] - 73s 229ms/step - loss: 1.0327 - auc: 0.8502 - acc: 0.5927 - val_loss: 1.4987 - val_auc: 0.7478 - val_acc: 0.4449
Epoch 3/50
321/321 [==============================] - 72s 224ms/step - loss: 0.9156 - auc: 0.8731 - acc: 0.6272 - val_loss: 1.5363 - val_auc: 0.7459 - val_acc: 0.4394
Epoch 4/50
321/321 [==============================] - 74s 230ms/step - loss: 0.9097 - auc: 0.8807 - acc: 0.6456 - val_loss: 1.4880 - val_auc: 0.7405 - val_acc: 0.4261
Epoch 5/50
321/321 [==============================] - 74s 232ms/step - loss: 0.7800 - auc: 0.9018 - acc: 0.6803 - val_loss: 1.3719 - val_auc: 0.7629 - val_acc: 0.4597
Epoch 6/50
321/321 [==============================] - 75s 233ms/step - loss: 0.8510 - auc: 0.8934 -

In [6]:
best_model = keras.models.load_model('best_model.h5')
loss, auc, accuracy = best_model.evaluate(validation_dataset)
print(f"Loss: {loss}, AUC: {auc}, Accuracy: {accuracy}")

80/80 [==============================] - 15s 182ms/step - loss: 1.2153 - auc: 0.7636 - acc: 0.4863
Loss: 1.2152652740478516, AUC: 0.7635571360588074, Accuracy: 0.4863174259662628


In [7]:
best_model.summary()

Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_4 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
vgg16 (Functional)           (None, 7, 7, 512)         14714688  
_________________________________________________________________
flatten_1 (Flatten)          (None, 25088)             0         
_________________________________________________________________
dense_1 (Dense)              (None, 4)                 100356    
Total params: 14,815,044
Trainable params: 100,356
Non-trainable params: 14,714,688
_________________________________________________________________


In [8]:
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    BatchNormalization
)

from tensorflow.keras import backend as K
from tensorflow.keras.layers import Layer

In [9]:
class FuzzyLayer(Layer):

    def __init__(self, output_dim, **kwargs):
        self.output_dim = output_dim
        super(FuzzyLayer, self).__init__(**kwargs)

    def build(self, input_shape):

        self.mu = self.add_weight(
            name='mu',
            shape=(input_shape[-1], self.output_dim),
            initializer='uniform',
            trainable=True
        )

        self.sigma = self.add_weight(
            name='sigma',
            shape=(input_shape[-1], self.output_dim),
            initializer='ones',
            trainable=True
        )

        super(FuzzyLayer, self).build(input_shape)

    def call(self, inputs):

        x = K.expand_dims(inputs, axis=-1)

        fuzzy_membership = K.exp(
            -K.square(x - self.mu) /
            (2 * K.square(self.sigma) + K.epsilon())
        )

        return K.mean(fuzzy_membership, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)

    # IMPORTANT FIX
    def get_config(self):

        config = super(FuzzyLayer, self).get_config()

        config.update({
            'output_dim': self.output_dim
        })

        return config

In [10]:
from tensorflow.keras.applications.vgg19 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.2,
    validation_split=0.2
)

In [11]:
train_dataset = train_datagen.flow_from_directory(
    directory='/kaggle/input/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/Alzheimer_s Dataset/train/',
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_dataset = train_datagen.flow_from_directory(
    directory='/kaggle/input/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/Alzheimer_s Dataset/train/',
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 5121 images belonging to 4 classes.
Found 1279 images belonging to 4 classes.


In [12]:
from sklearn.utils.class_weight import compute_class_weight

classes = train_dataset.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: 1.7855648535564854, 1: 24.620192307692307, 2: 0.50009765625, 3: 0.7144252232142857}


In [13]:
def train_fuzzy_model(base_model, train_data, val_data, epochs=50):

    # Fine tuning
    base_model.trainable = True

    for layer in base_model.layers[:-20]:
        layer.trainable = False

    inputs = Input(shape=(224, 224, 3))

    x = base_model(inputs, training=False)

    # Replace Flatten
    x = GlobalAveragePooling2D()(x)

    x = BatchNormalization()(x)

    x = Dense(256, activation='relu')(x)

    x = Dropout(0.4)(x)

    # FUZZY LAYER
    x = FuzzyLayer(128)(x)

    x = Dense(64, activation='relu')(x)

    x = Dropout(0.3)(x)

    outputs = Dense(4, activation='softmax')(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=[
            keras.metrics.AUC(name='auc'),
            'accuracy'
        ]
    )

    model.summary()

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        class_weight=class_weights
    )

    return model

In [14]:
vgg19_model = VGG19(
    input_shape=(224,224,3),
    weights='imagenet',
    include_top=False
)

model_vgg19_fuzzy = train_fuzzy_model(
    vgg19_model,
    train_dataset,
    validation_dataset
)

Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_8 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
vgg19 (Functional)           (None, 7, 7, 512)         20024384  
_________________________________________________________________
global_average_pooling2d (Gl (None, 512)               0         
_________________________________________________________________
batch_normalization (BatchNo (None, 512)               2048      
_________________________________________________________________
dense_3 (Dense)              (None, 256)               131328    
_________________________________________________________________
dropout (Dropout)            (None, 256)               0         
_________________________________________________________________
fuzzy_layer (FuzzyLayer)     (None, 128)               6553

In [15]:
model_vgg19_fuzzy.save('fuzzy_vgg19_model.h5')

In [16]:
loss, auc, accuracy = model_vgg19_fuzzy.evaluate(validation_dataset)

print("Loss:", loss)
print("AUC:", auc)
print("Accuracy:", accuracy)

80/80 [==============================] - 15s 184ms/step - loss: 1.0503 - auc: 0.8061 - accuracy: 0.5199
Loss: 1.050336480140686
AUC: 0.8061174154281616
Accuracy: 0.5199374556541443
